### 감사의견 수집

In [ ]:
import sqlite3
import pandas as pd
import requests
import time
import zipfile
import io
import os

# ── API 키 설정 (4개) ────────────────────────────────────────
API_KEYS = [
    'YOUR_API_KEY_1',
    'YOUR_API_KEY_2',
    'YOUR_API_KEY_3',
    'YOUR_API_KEY_4',
]
API_KEY = API_KEYS[0]   # 기존 단일 키 호환용

# ── 보고서 코드 ──────────────────────────────────────────────
# 11011: 사업보고서 (기존 수집 완료)
# 11012: 반기보고서 → 검토의견 존재 (수집 가치 있음)
# 11013: 1분기보고서 → 감사의견 없음 (대부분 NO_DATA)
# 11014: 3분기보고서 → 감사의견 없음 (대부분 NO_DATA)
EXTRA_REPORT_CODES = ['11012']   # 반기보고서만 수집 (실질 데이터 있음)
# EXTRA_REPORT_CODES = ['11012', '11013', '11014']  # 전체 수집 시 주석 해제

# ── 파일 경로 ──────────────────────────────────────────────
PROGRESS_FILE       = './data/audit_progress.csv'        # 사업보고서 (기존)
RESULT_FILE         = './data/audit_result.csv'          # 사업보고서 (기존)
EXTRA_PROGRESS_FILE = './data/audit_progress_extra.csv'  # 추가 보고서
EXTRA_RESULT_FILE   = './data/audit_result_extra.csv'    # 추가 보고서

In [ ]:
# (stock_code, year) 쌍 로드
def load_targets(db_path="../data/financial_db.db"):
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query("SELECT DISTINCT stock_code, year FROM financials_v2", conn)
    conn.close()
    df['stock_code'] = df['stock_code'].astype(str).str.zfill(6)
    df['year'] = df['year'].astype(int)
    return df

def load_corp_code_map(dart_key):
    url = f"https://opendart.fss.or.kr/api/corpCode.xml?crtfc_key={dart_key}"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
        z.extractall()
    corp_df = pd.read_xml("CORPCODE.xml")
    corp_df["corp_code"] = corp_df["corp_code"].astype(str).str.zfill(8)
    corp_df["stock_code"] = corp_df["stock_code"].astype(str).str.strip()
    return corp_df.set_index("stock_code")["corp_code"].to_dict()

# ── API 키 로테이션 관리자 ────────────────────────────────────
class APIKeyManager:
    def __init__(self, keys):
        self.keys = keys
        self.current_idx = 0
        self.exhausted = set()

    @property
    def current_key(self):
        return self.keys[self.current_idx]

    @property
    def all_exhausted(self):
        return len(self.exhausted) >= len(self.keys)

    def mark_exhausted_and_rotate(self):
        """현재 키를 소진 처리하고 다음 가용 키로 전환. 가용 키가 있으면 True 반환."""
        self.exhausted.add(self.current_idx)
        print(f"  [KEY {self.current_idx+1}/{len(self.keys)} 소진] 남은 키: {len(self.keys) - len(self.exhausted)}개")
        for _ in range(len(self.keys)):
            self.current_idx = (self.current_idx + 1) % len(self.keys)
            if self.current_idx not in self.exhausted:
                print(f"  → API 키 {self.current_idx+1} 전환")
                return True
        return False

# ── DART API 호출 ─────────────────────────────────────────────
def get_audit_opinion(dart_key, corp_code, year, reprt_code="11011"):
    resp = requests.get(
        "https://opendart.fss.or.kr/api/accnutAdtorNmNdAdtOpinion.json",
        params={
            "crtfc_key": dart_key,
            "corp_code": corp_code,
            "bsns_year": year,
            "reprt_code": reprt_code,
        },
        timeout=10,
    ).json()
    status = resp.get("status")
    # 020: API 호출 한도 초과
    if status == "020":
        return None, "RATE_LIMIT"
    if status == "000" and resp.get("list"):
        return resp["list"], "OK"
    return [], "NO_DATA"

def call_with_rotation(key_manager, corp_code, year, reprt_code):
    """RATE_LIMIT 발생 시 자동 키 로테이션 후 재시도."""
    while True:
        try:
            items, status = get_audit_opinion(key_manager.current_key, corp_code, year, reprt_code)
        except Exception:
            return None, "ERROR"
        if status != "RATE_LIMIT":
            return items, status
        has_next = key_manager.mark_exhausted_and_rotate()
        if not has_next:
            return None, "ALL_EXHAUSTED"

# ── 결과 정규화 ──────────────────────────────────────────────
def normalize_audit_items(stock_code, year, items, reprt_code=None):
    rows = []
    for item in items or []:
        rows.append({
            "stock_code": stock_code,
            "year": year,
            "reprt_code": reprt_code,
            "corp_name": item.get("corp_name"),
            "bsns_year": item.get("bsns_year"),
            "corp_cls": item.get("corp_cls"),
            "auditor": item.get("adtor"),
            "audit_opinion": (item.get("adt_opinion") or "").strip(),
            "adt_reprt_spcmnt_matter": item.get("adt_reprt_spcmnt_matter"),
            "emphasis": item.get("emphs_matter"),
            "core_audit_matter": item.get("core_adt_matter"),
            "stlm_dt": item.get("stlm_dt"),
        })
    return rows

# ── 진행 상태 초기화 ─────────────────────────────────────────
def init_progress(targets_df, stock_to_corp):
    """사업보고서용 기존 진행 파일 초기화 (하위 호환)."""
    if os.path.exists(PROGRESS_FILE):
        progress = pd.read_csv(PROGRESS_FILE, dtype={'stock_code': str})
        progress['stock_code'] = progress['stock_code'].str.zfill(6)
        print(f"기존 진행 파일 로드: {len(progress)}건 (완료: {(progress['status'] != 'PENDING').sum()}건)")
        return progress

    progress = targets_df.copy()
    progress['has_corp_code'] = progress['stock_code'].map(lambda x: x in stock_to_corp)
    progress['status'] = 'PENDING'
    progress.loc[~progress['has_corp_code'], 'status'] = 'NO_CORP_CODE'
    progress.drop(columns=['has_corp_code'], inplace=True)
    progress.to_csv(PROGRESS_FILE, index=False)

    pending = (progress['status'] == 'PENDING').sum()
    skip = (progress['status'] == 'NO_CORP_CODE').sum()
    print(f"진행 파일 생성: 전체 {len(progress)}건 / 수집대상 {pending}건 / corp_code 없음 {skip}건")
    return progress

def init_extra_progress(targets_df, stock_to_corp, report_codes=None):
    """추가 보고서(반기/1분기/3분기)용 진행 파일 초기화."""
    if report_codes is None:
        report_codes = EXTRA_REPORT_CODES

    if os.path.exists(EXTRA_PROGRESS_FILE):
        progress = pd.read_csv(EXTRA_PROGRESS_FILE, dtype={'stock_code': str, 'reprt_code': str})
        progress['stock_code'] = progress['stock_code'].str.zfill(6)
        done = (progress['status'] != 'PENDING').sum()
        pending = (progress['status'] == 'PENDING').sum()
        print(f"기존 추가 진행 파일 로드: {len(progress)}건 (완료: {done}건 / 남은: {pending}건)")
        return progress

    rows = [
        {'stock_code': row['stock_code'], 'year': row['year'], 'reprt_code': rc}
        for _, row in targets_df.iterrows()
        for rc in report_codes
    ]
    progress = pd.DataFrame(rows)
    progress['has_corp_code'] = progress['stock_code'].map(lambda x: x in stock_to_corp)
    progress['status'] = 'PENDING'
    progress.loc[~progress['has_corp_code'], 'status'] = 'NO_CORP_CODE'
    progress.drop(columns=['has_corp_code'], inplace=True)
    progress.to_csv(EXTRA_PROGRESS_FILE, index=False)

    pending = (progress['status'] == 'PENDING').sum()
    skip = (progress['status'] == 'NO_CORP_CODE').sum()
    print(f"추가 진행 파일 생성: 전체 {len(progress)}건 / 수집대상 {pending}건 / corp_code 없음 {skip}건")
    return progress

In [11]:
targets_df = load_targets()
stock_to_corp = load_corp_code_map(API_KEY)
progress = init_progress(targets_df, stock_to_corp)

pending = (progress['status'] == 'PENDING').sum()
print(f"\n남은 수집 대상: {pending}건")

진행 파일 생성: 전체 21994건 / 수집대상 21994건 / corp_code 없음 0건

남은 수집 대상: 21994건


In [12]:
def collect_audit_data(progress, stock_to_corp, save_interval=100):
    pending_mask = progress['status'] == 'PENDING'
    pending_rows = progress[pending_mask].index.tolist()
    total = len(pending_rows)
    
    if total == 0:
        print("수집할 대상이 없습니다.")
        return
    
    print(f"수집 시작: {total}건")
    
    collected_rows = []
    api_calls = 0
    
    for i, idx in enumerate(pending_rows):
        stock_code = progress.at[idx, 'stock_code']
        year = int(progress.at[idx, 'year'])
        corp_code = stock_to_corp.get(stock_code)
        
        try:
            items, status = get_audit_opinion(API_KEY, corp_code, year)
        except Exception as e:
            print(f"\n[ERROR] {stock_code}/{year}: {e}")
            progress.at[idx, 'status'] = 'ERROR'
            continue
        
        api_calls += 1
        
        # API 한도 초과 → 즉시 중단 + 저장
        if status == "RATE_LIMIT":
            print(f"\n[RATE LIMIT] API 호출 한도 초과! (호출 {api_calls}회)")
            print(f"진행률: {i}/{total} ({i/total*100:.1f}%)")
            _save_progress(progress, collected_rows)
            return
        
        # 정상 수집
        if status == "OK":
            collected_rows.extend(normalize_audit_items(stock_code, year, items))
            progress.at[idx, 'status'] = 'OK'
        else:
            progress.at[idx, 'status'] = 'NO_DATA'
        
        # 중간 저장
        if (i + 1) % save_interval == 0:
            _save_progress(progress, collected_rows)
            remaining = total - (i + 1)
            print(f"[{i+1}/{total}] 저장 완료 | 수집 {len(collected_rows)}건 | 남은 {remaining}건")
        
        time.sleep(0.5)  # API rate limit 방지
    
    # 최종 저장
    _save_progress(progress, collected_rows)
    print(f"\n수집 완료! 총 {len(collected_rows)}건 저장")


def _save_progress(progress, collected_rows):
    """진행 상태 + 결과 저장"""
    progress.to_csv(PROGRESS_FILE, index=False)
    
    if collected_rows:
        new_df = pd.DataFrame(collected_rows)
        if os.path.exists(RESULT_FILE):
            new_df.to_csv(RESULT_FILE, mode='a', header=False, index=False)
        else:
            new_df.to_csv(RESULT_FILE, index=False)
        collected_rows.clear()  # 저장 후 메모리 비우기

# 실행 (PENDING 상태인 것만 수집, 중단 후 재실행하면 이어서 수집)
collect_audit_data(progress, stock_to_corp)

수집 시작: 21994건
[100/21994] 저장 완료 | 수집 0건 | 남은 21894건
[200/21994] 저장 완료 | 수집 0건 | 남은 21794건
[300/21994] 저장 완료 | 수집 0건 | 남은 21694건
[400/21994] 저장 완료 | 수집 0건 | 남은 21594건
[500/21994] 저장 완료 | 수집 0건 | 남은 21494건
[600/21994] 저장 완료 | 수집 0건 | 남은 21394건
[700/21994] 저장 완료 | 수집 0건 | 남은 21294건
[800/21994] 저장 완료 | 수집 0건 | 남은 21194건
[900/21994] 저장 완료 | 수집 0건 | 남은 21094건
[1000/21994] 저장 완료 | 수집 0건 | 남은 20994건
[1100/21994] 저장 완료 | 수집 0건 | 남은 20894건
[1200/21994] 저장 완료 | 수집 0건 | 남은 20794건
[1300/21994] 저장 완료 | 수집 0건 | 남은 20694건
[1400/21994] 저장 완료 | 수집 0건 | 남은 20594건
[1500/21994] 저장 완료 | 수집 0건 | 남은 20494건
[1600/21994] 저장 완료 | 수집 0건 | 남은 20394건
[1700/21994] 저장 완료 | 수집 0건 | 남은 20294건
[1800/21994] 저장 완료 | 수집 0건 | 남은 20194건
[1900/21994] 저장 완료 | 수집 0건 | 남은 20094건
[2000/21994] 저장 완료 | 수집 0건 | 남은 19994건
[2100/21994] 저장 완료 | 수집 0건 | 남은 19894건
[2200/21994] 저장 완료 | 수집 0건 | 남은 19794건
[2300/21994] 저장 완료 | 수집 0건 | 남은 19694건
[2400/21994] 저장 완료 | 수집 0건 | 남은 19594건
[2500/21994] 저장 완료 | 수집 0건 | 남은 19494건
[2600/21994] 저장 완료 |

In [2]:
# 수집 현황 확인
progress = pd.read_csv(PROGRESS_FILE, dtype={'stock_code': str})
print("[진행 상태 요약]")
print(progress['status'].value_counts().to_string())

if os.path.exists(RESULT_FILE):
    result = pd.read_csv(RESULT_FILE, dtype={'stock_code': str})
    print(f"\n[수집 결과] {len(result)}건 / {result['stock_code'].nunique()}개 종목")
    print(result.head())

[진행 상태 요약]
status
OK         19285
NO_DATA     2709

[수집 결과] 66259건 / 2756개 종목
  stock_code  year corp_name  bsns_year corp_cls auditor audit_opinion  \
0     372800  2021    아이티아이즈   제10기(당기)        K  대주회계법인            적정   
1     372800  2021    아이티아이즈    제9기(전기)        K  안진회계법인            적정   
2     372800  2021    아이티아이즈   제8기(전전기)        K  대주회계법인            적정   
3     372800  2022    아이티아이즈   제11기(당기)        K  대주회계법인            적정   
4     372800  2022    아이티아이즈   제10기(전기)        K  대주회계법인            적정   

  adt_reprt_spcmnt_matter emphasis core_audit_matter     stlm_dt  
0                       -  해당사항 없음               NaN  2021-12-31  
1                       -  해당사항 없음               NaN  2021-12-31  
2                       -  해당사항 없음               NaN  2021-12-31  
3                       -  해당사항 없음              (주1)  2022-12-31  
4                       -  해당사항 없음              (주2)  2022-12-31  


In [3]:
result

,stock_code,year,corp_name,bsns_year,corp_cls,auditor,audit_opinion,adt_reprt_spcmnt_matter,emphasis,core_audit_matter,stlm_dt
0,372800,2021,아이티아이즈,제10기(당기),K,대주회계법인,적정,-,해당사항 없음,NaN,2021-12-31
1,372800,2021,아이티아이즈,제9기(전기),K,안진회계법인,적정,-,해당사항 없음,NaN,2021-12-31
2,372800,2021,아이티아이즈,제8기(전전기),K,대주회계법인,적정,-,해당사항 없음,NaN,2021-12-31
3,372800,2022,아이티아이즈,제11기(당기),K,대주회계법인,적정,-,해당사항 없음,(주1),2022-12-31
4,372800,2022,아이티아이즈,제10기(전기),K,대주회계법인,적정,-,해당사항 없음,(주2),2022-12-31
...,...,...,...,...,...,...,...,...,...,...,...
66254,000210,2024,DL,제78기\n(당기),Y,삼일회계법인,적정의견,-,해당사항 없음,DL이앤씨 관계기업투자주식에 대한 손상평가,2024-12-31
66255,000210,2024,DL,제77기\n(전기),Y,삼일회계법인,적정의견,-,해당사항 없음,DL이앤씨 관계기업투자주식에 대한 손상평가,2024-12-31
66256,000210,2024,DL,제77기\n(전기),Y,삼일회계법인,적정의견,-,전기 연결재무제표 재작성,DL이앤씨 관계기업투자주식에 대한 손상평가,2024-12-31
66257,000210,2024,DL,제76기\n(전전기),Y,안진회계법인,적정의견,-,재무제표 재작성,DL이앤씨 관계기업투자주식에 대한 손상평가,2024-12-31


## 추가 보고서 수집 (반기 / 1분기 / 3분기)

사업보고서(11011) 이외의 보고서에서 감사의견을 추가 수집합니다.  
API 키 4개를 순차적으로 사용하며, 한도 초과 시 자동으로 다음 키로 전환합니다.

| 보고서 코드 | 보고서명 | 비고 |
|---|---|---|
| 11012 | 반기보고서 | 검토의견 포함 |
| 11013 | 1분기보고서 | 데이터 없을 수 있음 |
| 11014 | 3분기보고서 | 데이터 없을 수 있음 |

In [ ]:
# ── 추가 보고서 진행 초기화 ────────────────────────────────────
# targets_df, stock_to_corp는 위 섹션에서 이미 로드된 변수를 재사용
key_manager = APIKeyManager(API_KEYS)
extra_progress = init_extra_progress(targets_df, stock_to_corp)

pending = (extra_progress['status'] == 'PENDING').sum()
print(f"\n남은 수집 대상: {pending}건")
print(f"보고서 종류: {EXTRA_REPORT_CODES}")
print(f"사용 API 키: {len(API_KEYS)}개")

In [ ]:
def _save_extra(progress, collected_rows):
    """추가 보고서 진행 상태 + 결과 저장."""
    progress.to_csv(EXTRA_PROGRESS_FILE, index=False)
    if collected_rows:
        new_df = pd.DataFrame(collected_rows)
        if os.path.exists(EXTRA_RESULT_FILE):
            new_df.to_csv(EXTRA_RESULT_FILE, mode='a', header=False, index=False)
        else:
            new_df.to_csv(EXTRA_RESULT_FILE, index=False)
        collected_rows.clear()


def collect_extra_audit_data(progress, stock_to_corp, key_manager, save_interval=100):
    pending_mask = progress['status'] == 'PENDING'
    pending_rows = progress[pending_mask].index.tolist()
    total = len(pending_rows)

    if total == 0:
        print("수집할 대상이 없습니다.")
        return

    print(f"수집 시작: {total}건 / 사용 가능 API 키: {len(key_manager.keys) - len(key_manager.exhausted)}개")

    collected_rows = []
    api_calls = 0

    for i, idx in enumerate(pending_rows):
        stock_code = progress.at[idx, 'stock_code']
        year = int(progress.at[idx, 'year'])
        reprt_code = str(progress.at[idx, 'reprt_code'])
        corp_code = stock_to_corp.get(stock_code)

        items, status = call_with_rotation(key_manager, corp_code, year, reprt_code)
        api_calls += 1

        if status == "ALL_EXHAUSTED":
            print(f"\n[ALL KEYS EXHAUSTED] 모든 API 키 소진! (총 호출 {api_calls}회)")
            print(f"진행률: {i}/{total} ({i/total*100:.1f}%)")
            _save_extra(progress, collected_rows)
            return

        if status == "ERROR":
            progress.at[idx, 'status'] = 'ERROR'
        elif status == "OK":
            collected_rows.extend(normalize_audit_items(stock_code, year, items, reprt_code))
            progress.at[idx, 'status'] = 'OK'
        else:  # NO_DATA
            progress.at[idx, 'status'] = 'NO_DATA'

        # 중간 저장
        if (i + 1) % save_interval == 0:
            _save_extra(progress, collected_rows)
            remaining = total - (i + 1)
            print(f"[{i+1}/{total}] 저장 완료 | 수집 {len(collected_rows)}건 | 남은 {remaining}건 | 현재 키: {key_manager.current_idx+1}")

        time.sleep(0.5)

    _save_extra(progress, collected_rows)
    print(f"\n수집 완료! 총 {api_calls}회 호출")


# ── 실행 (PENDING 상태만 수집, 중단 후 재실행하면 이어서 수집) ──
collect_extra_audit_data(extra_progress, stock_to_corp, key_manager)

In [ ]:
# ── 추가 보고서 수집 현황 확인 ──────────────────────────────────
extra_progress = pd.read_csv(EXTRA_PROGRESS_FILE, dtype={'stock_code': str, 'reprt_code': str})

print("[추가 보고서 진행 상태 요약]")
summary = extra_progress.groupby(['reprt_code', 'status']).size().unstack(fill_value=0)
print(summary.to_string())

if os.path.exists(EXTRA_RESULT_FILE):
    result = pd.read_csv(EXTRA_RESULT_FILE, dtype={'stock_code': str, 'reprt_code': str})
    print(f"\n[수집 결과] {len(result)}건 / {result['stock_code'].nunique()}개 종목")
    print(result.groupby('reprt_code').size().rename('수집건수'))
    print(result.head())

In [ ]:
# ── 사업보고서 + 추가 보고서 병합 ────────────────────────────────────────
result_df = pd.read_csv(RESULT_FILE, dtype={'stock_code': str})
extra_result_df = pd.read_csv(EXTRA_RESULT_FILE, dtype={'stock_code': str, 'reprt_code': str})

# stock_code 6자리 패딩
result_df['stock_code'] = result_df['stock_code'].str.zfill(6)
extra_result_df['stock_code'] = extra_result_df['stock_code'].str.zfill(6)

# stlm_dt 날짜 변환
result_df['stlm_dt'] = pd.to_datetime(result_df['stlm_dt'], errors='coerce')
extra_result_df['stlm_dt'] = pd.to_datetime(extra_result_df['stlm_dt'], errors='coerce')

# stock별 stlm_dt 순 정렬 후 concat (extra_result_df는 reprt_code 제외)
result_df = result_df.sort_values(['stock_code', 'stlm_dt']).reset_index(drop=True)
extra_result_df = (extra_result_df.drop(columns=['reprt_code'])
                                  .sort_values(['stock_code', 'stlm_dt'])
                                  .reset_index(drop=True))

audit_all = (pd.concat([result_df, extra_result_df], ignore_index=True)
               .sort_values(['stock_code', 'stlm_dt'])
               .reset_index(drop=True))

print(f'사업보고서  : {len(result_df):,}건')
print(f'추가 보고서 : {len(extra_result_df):,}건')
print(f'전체 합계   : {len(audit_all):,}건 / {audit_all["stock_code"].nunique():,}개 종목')
audit_all.head(10)